# Polytope examples & plots (2D + 3D)

This notebook exercises every linear primitive via `Deferred(...)` on a `TVHJImpl`
grid, checks numeric agreement with reference builders, and visualizes
each shape in **2D** and **3D**:

- **2D** — heatmap on `(x, y)` at `t = 0` (`method="2D"`)
- **3D** — isosurface on `(x, y, yaw)` at `t = 0` (`method="3D"`)


| `Deferred(...)` method | Shape |
|------------------------|-------|
| `'halfspace'` | one half-space |
| `'slab'` | band between two parallel planes |
| `'box'` | axis-aligned rectangle in `(x, y)` |
| `'polygon'` | regular hexagon in `(x, y)` |
| `'from_vertices'` | convex polygon from corner list |


In [5]:
from math import pi

import numpy as np

from pyspect import *
from pyspect.impls.hj_reachability import TVHJImpl
from pyspect.systems.hj_reachability import Air3d

AXES = [
    dict(name="t", bounds=[0, 1], step=1.0, unit="s"),
    dict(name="x", bounds=[-5, +5], points=25, unit="m"),
    dict(name="y", bounds=[-5, +5], points=25, unit="m"),
    dict(name="yaw", bounds=[-pi, +pi], points=12, unit="rad"),
]

impl = TVHJImpl(dict(cls=Air3d), AXES)
SLICE_T = [("t", 0)]
CAMERA_3D = impl.PLOT.EYE_ML_SW


def maxdiff(a, b) -> float:
    return float(np.max(np.abs(np.asarray(a) - np.asarray(b))))


def show_2d(out, title: str, *, colorscale: str = "Greens") -> None:
    """Heatmap on (x, y) at t = 0."""
    impl.plot(
        out,
        method="2D",
        axes=("x", "y"),
        transform_select=SLICE_T,
        layout_title=f"[2D] {title}",
        colorscale=colorscale,
        layout_height=360,
        layout_width=480,
    ).show()


def show_3d(out, title: str, *, colorscale: str = "Greens") -> None:
    """Isosurface on (x, y, yaw) at t = 0."""
    impl.plot(
        out,
        method="3D",
        axes=("x", "y", "yaw"),
        transform_select=SLICE_T,
        layout_title=f"[3D] {title}",
        colorscale=colorscale,
        camera_eye=CAMERA_3D,
        layout_height=480,
        layout_width=640,
    ).show()

## 1. `Deferred('halfspace', ...)` — x ≥ 0

In 3D the constraint is unchanged in `yaw` (the half-space extends through the heading axis).

In [6]:
S = HalfSpaceSet(normal=[1, 0], offset=[0, 0], axes=["x", "y"])
ref = HalfSpaceSet(normal=[1, 0], offset=[0, 0], axes=["x", "y"])

out = S(impl)
print(f"max |halfspace - halfspace| = {maxdiff(out, ref(impl))}")
show_2d(out, "halfspace: x ≥ 0", colorscale="Blues")
show_3d(out, "halfspace: x ≥ 0", colorscale="Blues")

max |halfspace - halfspace| = 0.0


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

## 2. `Deferred('slab', ...)` — band −1 ≤ x ≤ 1

In [ ]:
S = Deferred('slab', axes=["x", "y"], normal=[1, 0], offset=[-1, 0], width=2)
ref = Inter(
    Deferred('halfspace', normal=[1, 0], offset=[-1, 0], axes=["x", "y"]),
    Deferred('halfspace', normal=[-1, 0], offset=[1, 0], axes=["x", "y"]),
)

out = S(impl)
print(f"max |slab - Inter(H1,H2)| = {maxdiff(out, ref(impl))}")
show_2d(out, "slab: −1 ≤ x ≤ 1", colorscale="Oranges")
show_3d(out, "slab: −1 ≤ x ≤ 1", colorscale="Oranges")

## 3. `Deferred('box', ...)` — rectangle x ∈ [−2, 2], y ∈ [−1, 1]

Only `(x, y)` are bounded; in 3D the shape is the rectangle extruded along `yaw`.

In [ ]:
S = Deferred('box', x=(-2, 2), y=(-1, 1))

out = S(impl)
show_2d(out, "box: x∈[−2,2], y∈[−1,1]", colorscale="Greens")
show_3d(out, "box: x∈[−2,2], y∈[−1,1] (extruded in yaw)", colorscale="Greens")

## 4. `Deferred('polygon', ...)` — regular hexagon, radius 2

In [ ]:
import math
from functools import reduce

order, radius = 6, 2.0
S = Deferred('polygon', order=order, radius=radius, axes=["x", "y"])

vertices = [
    [radius * math.cos(2 * math.pi * k / order), radius * math.sin(2 * math.pi * k / order)]
    for k in range(order)
]
walls = []
for k in range(order):
    v0 = vertices[k]
    v1 = vertices[(k + 1) % order]
    dx, dy = v1[0] - v0[0], v1[1] - v0[1]
    walls.append(Deferred('halfspace', normal=[-dy, dx], offset=v0, axes=["x", "y"]))
ref = reduce(Inter, walls)

out = S(impl)
print(f"max |polygon - manual hex| = {maxdiff(out, ref(impl))}")
show_2d(out, "polygon: regular hexagon, r=2", colorscale="Purples")
show_3d(out, "polygon: hexagon extruded in yaw", colorscale="Purples")

## 5. `Deferred('from_vertices', ...)` — irregular quadrilateral (trapezoid)

**What it represents:** any **convex polygon in the (x, y) plane**, defined by listing its
corners in **counter-clockwise (CCW) order**. Each edge becomes one inward-facing half-space;
`'from_vertices'` is the general case unlike `'polygon'` (regular only) or `'box'` (axis-aligned only).


In [ ]:
from functools import reduce

trap = [[-2, -1], [2, -1], [1, 1], [-1, 1]]
S = Deferred('from_vertices', vertices=trap, axes=["x", "y"])

walls = []
for k in range(len(trap)):
    v0 = trap[k]
    v1 = trap[(k + 1) % len(trap)]
    dx, dy = v1[0] - v0[0], v1[1] - v0[1]
    walls.append(Deferred('halfspace', normal=[-dy, dx], offset=v0, axes=["x", "y"]))
ref = reduce(Inter, walls)

out = S(impl)
print(f"max |from_vertices - manual Inter| = {maxdiff(out, ref(impl))}")
show_2d(out, "from_vertices: trapezoid", colorscale="Teal")
show_3d(out, "trapezoid extruded in yaw", colorscale="Teal")